# VAIC2026 — Đo lại trên tập TEST MỞ RỘNG (33 câu)

Lần trước tập test chỉ 18 câu nên **chạm trần**: Recall@3/@5 đã = 1.000, chỉ 2 câu sai
hạng-1, mỗi câu đáng 5,6 điểm phần trăm — không đủ phân giải để thấy cải thiện nhỏ.

Đã soạn thêm 26 câu hỏi Điện Biên từ các chunk chưa dùng → 79 QA, chia lại **grouped
theo positive chunk** (không rò rỉ): **46 train / 33 test**. Mỗi câu giờ = 3,0 điểm.

Chạy **cả hai cấu hình trên cùng tập test mới** để so sánh vẫn có kiểm soát:
- A: LoRA train **46 câu Điện Biên** (baseline mới)
- B: LoRA train **78 câu** = 46 Điện Biên + 32 Sơn La/Đắk Lắk


In [ ]:
# Kaggle's torch (2.10+cu128) dropped Pascal sm_60 but the assigned GPU is often a
# P100 (sm_60) -> "no kernel image". Install a torch that supports P100 AND T4.
# MUST run before any `import torch`. Also drop torchvision/torchaudio (mismatched
# -> "torchvision::nms does not exist") and torchao (peft rejects Kaggle's 0.10.0).
!pip uninstall -y -q torchvision torchaudio torchao
!pip install -q rank-bm25 peft
!pip install -q torch==2.6.0+cu124 --extra-index-url https://download.pytorch.org/whl/cu124

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda)
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0), '| sm_', torch.cuda.get_device_capability(0))
    x = torch.randn(8, device='cuda'); print('CUDA op OK:', float((x + 1).sum()))

In [ ]:
import glob

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} not found under /kaggle/input'
    return hits[0]

TRAIN_PY  = find('train_reranker.py')
EVAL_PY   = find('eval_reranker.py')
CHUNKS    = find('chunks.jsonl')
EVAL_33   = find('eval_dienbien_v2.jsonl')        # 33 câu Điện Biên, ĐÓNG BĂNG
TRAIN_A   = find('train_dienbien_v2.jsonl')       # 46 câu Điện Biên
TRAIN_B   = find('train_multiprovince_v3.jsonl')  # 46 ĐB + 32 tỉnh khác = 78
OUT_A = '/kaggle/working/lora-dienbien46'
OUT_B = '/kaggle/working/lora-multi78'
for k in ['TRAIN_PY','EVAL_PY','CHUNKS','EVAL_33','TRAIN_A','TRAIN_B']:
    print(k, '=', eval(k))

# xác nhận đúng số dòng (đừng tin tên file — tin số đếm)
for k, p in [('TRAIN_A', TRAIN_A), ('TRAIN_B', TRAIN_B), ('EVAL_33', EVAL_33)]:
    print(k, 'rows =', sum(1 for l in open(p, encoding='utf-8') if l.strip()))

## A — LoRA train 46 câu Điện Biên (baseline mới)

In [ ]:
!python {TRAIN_PY}     --skip-zalo --use-lora --lora-r 16 --lora-alpha 32     --base-model BAAI/bge-reranker-v2-m3     --indomain {TRAIN_A} --out {OUT_A}     --epochs 6 --batch-size 8 --grad-accum 1 --lr 1e-4 --max-len 384 --num-workers 2

In [ ]:
!python {EVAL_PY} --chunks {CHUNKS} --eval {EVAL_33}     --base-model BAAI/bge-reranker-v2-m3 --finetuned {OUT_A}
import shutil; shutil.copy('eval_results.json', '/kaggle/working/eval_A_dienbien46.json')

## B — LoRA train 78 câu đa tỉnh (46 ĐB + 32 tỉnh khác)
Cùng siêu tham số với A; **chỉ đổi dữ liệu train**.

In [ ]:
!python {TRAIN_PY}     --skip-zalo --use-lora --lora-r 16 --lora-alpha 32     --base-model BAAI/bge-reranker-v2-m3     --indomain {TRAIN_B} --out {OUT_B}     --epochs 6 --batch-size 8 --grad-accum 1 --lr 1e-4 --max-len 384 --num-workers 2

In [ ]:
!python {EVAL_PY} --chunks {CHUNKS} --eval {EVAL_33}     --base-model BAAI/bge-reranker-v2-m3 --finetuned {OUT_B}
import shutil; shutil.copy('eval_results.json', '/kaggle/working/eval_B_multi78.json')

## So sánh

In [ ]:
import json
A = json.load(open('/kaggle/working/eval_A_dienbien46.json'))
B = json.load(open('/kaggle/working/eval_B_multi78.json'))
print('=== A: 46 Dien Bien ==='); print(json.dumps(A, indent=2, ensure_ascii=False))
print('=== B: 78 multi ===');     print(json.dumps(B, indent=2, ensure_ascii=False))